# PatchTST and iTransformer Split-Horizon Forecasting, Leakage-Safe Selector (mixed5)

This notebook uses one shared BasicTS forecasting data pipeline for both models. PatchTST forecasts steps 1-6, iTransformer forecasts steps 7-12, and their 6-step outputs are concatenated into one 12-step hybrid forecast.

The selector section avoids test-set leakage: hard selector masks and soft weights are learned from validation predictions, then applied once to the test predictions for final reporting.

## 1. Project Setup

This cell keeps the notebook runnable from inside `notebooks/` by moving to the repo root and adding `src/` to Python's import path.


In [ ]:
import os 
import sys
from pathlib import Path
#root contains the path to the basicts
ROOT = Path(r"C:\Users\luwil\OneDrive\Documents\Code\BasicTS")
#move python working folder
os.chdir(ROOT)
# the path to src is src_path
src_path = ROOT / "src"
#if src_path is not in the system path, add it to the system path
#system path is added to python search. 
# This allows us to import modules from the src 
# folder without having to specify the full path.
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

## 2. Imports and Shared Settings

Both models use the same dataset, scaler, preprocessing, `input_len`, train/val/test split, and batch format. The shared dataset keeps the full 12-step target window, while each split-horizon model trains on its own 6-step slice.


In [ ]:
import json
from datetime import datetime
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader

from basicts.configs import BasicTSForecastingConfig, BasicTSModelConfig
from basicts.launcher import BasicTSLauncher
from basicts.models.PatchTST import PatchTSTConfig, PatchTSTForForecasting
from basicts.models.iTransformer import iTransformerConfig, iTransformerForForecasting
from basicts.runners.builder import Builder
from basicts.runners.taskflow import BasicTSForecastingTaskFlow
from basicts.scaler import ZScoreScaler
from basicts.utils import BasicTSMode
DATASET_NAME = "ETTh1"
#most papers use 96 input length
INPUT_LEN = 96
#most papers use 96, 192, 336, and 720 input length
FULL_OUTPUT_LEN = 12

SPLIT_OUTPUT_LEN = 6
#etthl has 7 variables
NUM_FEATURES = 7
#Batch sizes like 16, 32, and 64 are normal. 32 is a safe default.
BATCH_SIZE = 32
# common for testing
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3
# Fresh namespace for this notebook so BasicTS does not auto-resume from old/corrupt checkpoints.
RUN_TAG = "mixed_5"
# this is the shared settings dictionary that both models use
SHARED_CONFIG = {
    "dataset_name": DATASET_NAME,
    "input_len": INPUT_LEN,
    "dataset_params": {
        "input_len": INPUT_LEN,
        "output_len": FULL_OUTPUT_LEN,
        "use_timestamps": False,
        "memmap": False,
    },
    "use_timestamps": False,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "scaler": ZScoreScaler,
    "norm_each_channel": True,
    "rescale": False,
    "metrics": ["MAE", "MSE"],
    "optimizer_params": {"lr": LEARNING_RATE, "weight_decay": 5e-4},
    "gpus": None,
    "train_data_num_workers": 0,
    "val_data_num_workers": 0,
    "test_data_num_workers": 0,
    "save_results": True,
}

# The standalone 12-step models only need BasicTS test_metrics.json.
# Metrics-only evaluation avoids Windows memmap file-lock issues.
FULL_12_STEP_CONFIG = dict(SHARED_CONFIG)
FULL_12_STEP_CONFIG["save_results"] = False


def fresh_checkpoint_dir(model_folder, run_name):
    # Each training launch gets a unique parent folder, so BasicTS cannot resume a stale checkpoint.
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return str(Path("checkpoints") / RUN_TAG / model_folder / run_name / stamp)

## 3. Shared Shape Check Helper

This helper builds the BasicTS dataset and scaler, runs the same forecasting preprocessing that training uses, and then sends one batch through the selected model.


In [ ]:
# for the split forecasitting model, we need to create a custom taskflow that slices the targets and target masks to the desired output length      
#start with the forecasting taskflwo and mofify it 
# BasicTSForecastingTaskFlow is the default data-prep worker.
# It prepares each forecasting batch before the model uses it.
# start with the deault taskflow and then add a change 
class SplitHorizonForecastingTaskFlow(BasicTSForecastingTaskFlow):
    #adding a setting called targest slide whchi is a variable that is used to slice the targets
    # tells the taskflow which targets to keep
    # need self because we need acresss to this specific object
    #creates a variables incide the class object 
    def __init__(self, target_slice):
        self.target_slice = target_slice
    # preprocess 
    def preprocess(self, runner, data):
        #normal work
        data = super().preprocess(runner, data)
        #cuts target values
        data["targets"] = data["targets"][:, self.target_slice, :]
        # tells basicts which target values are valid
        data["targets_mask"] = data["targets_mask"][:, self.target_slice, :]
        return data


def _float_batch(batch):
    return {
        key: value.float() if isinstance(value, torch.Tensor) and value.is_floating_point() else value
        for key, value in batch.items()
    }

# checks to see if the inputs are the right shape the targest are sliced and the prediction matches targer
def preview_shapes(cfg, model_name):
    #build the dataset using basicts
    train_dataset = Builder._build_dataset(cfg, BasicTSMode.TRAIN)
    # puts the dataset into batches gets ready for batches
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=False)
    #This grabs the first batch.
    raw_batch = _float_batch(next(iter(train_loader)))
    #creates the scaler and fits it to the training data  
    scaler = Builder._build_scaler(cfg)
    scaler.fit(train_dataset.data)
    # fake runner so the pasicts 
    class PreviewRunner:
        pass
    # Create a fake runner that has cfg and scaler, because taskflow.preprocess expects a runner object.
    runner = PreviewRunner() 
    runner.cfg = cfg
    runner.scaler = scaler
    #prepares the batch normally
    processed_batch = cfg.taskflow.preprocess(runner, dict(raw_batch))
    #build the model from the config and switch it to evaluation mode
    model = cfg.model(cfg.model_config)
    model.eval()
    # o not track gradient as they are only needed for trainig 
    with torch.no_grad():
        #sends processed inputs into the model
        prediction = model(processed_batch["inputs"])
        # then the model outputs future values
        # did the model return a dictionary
        if isinstance(prediction, dict):
            # if it did this extracts onlt the prediction tensor
            prediction = prediction["prediction"]
    #print the shapres to check that the data and model match before the training
    print(f"{model_name} raw inputs shape:       ", tuple(raw_batch["inputs"].shape))
    print(f"{model_name} raw target shape:       ", tuple(raw_batch["targets"].shape))
    print(f"{model_name} processed inputs shape: ", tuple(processed_batch["inputs"].shape))
    print(f"{model_name} target shape:           ", tuple(processed_batch["targets"].shape))
    print(f"{model_name} prediction shape:       ", tuple(prediction.shape))
    #checks
    assert tuple(raw_batch["targets"].shape) == (cfg.batch_size, FULL_OUTPUT_LEN, NUM_FEATURES)
    assert tuple(processed_batch["inputs"].shape) == (cfg.batch_size, INPUT_LEN, NUM_FEATURES)
    assert tuple(processed_batch["targets"].shape) == (cfg.batch_size, SPLIT_OUTPUT_LEN, NUM_FEATURES)
    assert tuple(prediction.shape) == (cfg.batch_size, SPLIT_OUTPUT_LEN, NUM_FEATURES)
    return processed_batch, prediction


## 4. PatchTST Model

This notebook uses BasicTS's built-in `PatchTST` model for forecast steps 1-6. `PatchTST` receives `[batch_size, input_len, num_features]` and returns `[batch_size, 6, num_features]` for the split-horizon branch.

In [ ]:
# PatchTST is imported from BasicTS in the imports cell:
# from basicts.models.PatchTST import PatchTSTConfig, PatchTSTForForecasting
# No custom model class is needed for this branch.

## 5. PatchTST Config

This config reuses `BasicTSForecastingConfig` and only changes the model-specific pieces. The shared dataset/scaler/preprocessing settings come from `SHARED_CONFIG`.


In [ ]:
# tells BasicTS how to build and train PatchTST
patchtst_model_config = PatchTSTConfig(
    input_len=INPUT_LEN,
    output_len=SPLIT_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    patch_len=16,
    patch_stride=8,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    attn_dropout=0.1,
    fc_dropout=0.1,
    head_dropout=0.0,
    use_revin=True,
)

# full BasicTS training config for split-horizon PatchTST
patchtst_cfg = BasicTSForecastingConfig(
    model=PatchTSTForForecasting,
    model_config=patchtst_model_config,
    taskflow=SplitHorizonForecastingTaskFlow(slice(0, SPLIT_OUTPUT_LEN)),
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/PatchTSTForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_1_6",
    **SHARED_CONFIG,
)

# Standalone PatchTST trained to forecast all 12 steps.
patchtst_full_model_config = PatchTSTConfig(
    input_len=INPUT_LEN,
    output_len=FULL_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    patch_len=16,
    patch_stride=8,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    attn_dropout=0.1,
    fc_dropout=0.1,
    head_dropout=0.0,
    use_revin=True,
)

patchtst_full_cfg = BasicTSForecastingConfig(
    model=PatchTSTForForecasting,
    model_config=patchtst_full_model_config,
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/PatchTSTForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **FULL_12_STEP_CONFIG,
)

patchtst_cfg, patchtst_full_cfg


## 6. PatchTST Shape Test

Run this before training. The model prediction and processed target lines must be `(batch_size, 6, num_features)`, while the raw target line remains `(batch_size, 12, num_features)`.


In [ ]:
#checker
patchtst_batch, patchtst_prediction = preview_shapes(patchtst_cfg, "PatchTST")


## 7. Train the PatchTST

This is the first training run. Leave `RUN_PATCHTST_TRAINING` as `False` while editing or shape-checking, then switch it to `True` when you are ready to train.


In [ ]:
#training
RUN_PATCHTST_TRAINING = False
RUN_PATCHTST_12_STEP_TRAINING = False

if RUN_PATCHTST_TRAINING:
    patchtst_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "PatchTSTForForecasting",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_6",
    )
    print("Training split PatchTST in:", patchtst_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(patchtst_cfg)
else:
    print("PatchTST split-horizon training skipped. Set RUN_PATCHTST_TRAINING = True to train.")

if RUN_PATCHTST_12_STEP_TRAINING:
    patchtst_full_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "PatchTSTForForecasting",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    )
    print("Training 12-step PatchTST in:", patchtst_full_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(patchtst_full_cfg)
else:
    print("PatchTST 12-step training skipped. Set RUN_PATCHTST_12_STEP_TRAINING = True to train.")


## 8. iTransformer Model

After the PatchTST shape check works, use the repo's existing `iTransformerForForecasting`. It receives the same `[batch_size, input_len, num_features]` input and returns `[batch_size, 6, num_features]`. In this split-horizon hybrid, the iTransformer is responsible for forecast steps 7-12.


In [ ]:
#transfoemr model build it
transformer_model_config = iTransformerConfig(
    input_len=INPUT_LEN,
    output_len=SPLIT_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
    use_revin=True,
)

transformer_cfg = BasicTSForecastingConfig(
    model=iTransformerForForecasting,
    model_config=transformer_model_config,
    taskflow=SplitHorizonForecastingTaskFlow(slice(SPLIT_OUTPUT_LEN, FULL_OUTPUT_LEN)),
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/iTransformerForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_7_12",
    **SHARED_CONFIG,
)

# Standalone iTransformer trained to forecast all 12 steps.
transformer_full_model_config = iTransformerConfig(
    input_len=INPUT_LEN,
    output_len=FULL_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
    use_revin=True,
)

transformer_full_cfg = BasicTSForecastingConfig(
    model=iTransformerForForecasting,
    model_config=transformer_full_model_config,
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/iTransformerForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **FULL_12_STEP_CONFIG,
)

transformer_cfg, transformer_full_cfg


## 9. iTransformer Shape Test

Run this after the PatchTST section works. It uses the same shared data pipeline and checks that the iTransformer predicts only its 6-step split-horizon target.


In [ ]:
#check the shapes
transformer_batch, transformer_prediction = preview_shapes(transformer_cfg, "iTransformer")


## 10. Train the iTransformer

Use the same dataset, scaler, preprocessing, and `input_len` as the PatchTST. The shared dataset still contains the full 12-step target window, but the iTransformer taskflow slices that target to steps 7-12.


In [ ]:
#train trasnfoemr
RUN_TRANSFORMER_TRAINING = False
RUN_TRANSFORMER_12_STEP_TRAINING = False

if RUN_TRANSFORMER_TRAINING:
    transformer_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "iTransformerForForecasting",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_7_12",
    )
    print("Training split iTransformer in:", transformer_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(transformer_cfg)
else:
    print("iTransformer split-horizon training skipped. Set RUN_TRANSFORMER_TRAINING = True after the PatchTST works.")

if RUN_TRANSFORMER_12_STEP_TRAINING:
    transformer_full_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "iTransformerForForecasting",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    )
    print("Training 12-step iTransformer in:", transformer_full_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(transformer_full_cfg)
else:
    print("iTransformer 12-step training skipped. Set RUN_TRANSFORMER_12_STEP_TRAINING = True to train.")


## 11. Hybrid Prediction: PatchTST Steps 1-6, iTransformer Steps 7-12

This is a true split-horizon hybrid. The PatchTST predicts only the first 6 forecast steps, the iTransformer predicts only the next 6 forecast steps from the same input window, and the final hybrid prediction is the time-axis concatenation of those two 6-step outputs.


In [ ]:
import numpy as np

#checks that the patchtst and transformer both used the same batch
assert torch.equal(patchtst_batch["inputs"], transformer_batch["inputs"])

# Split-horizon hybrid: PatchTST predicts steps 1-6, iTransformer predicts steps 7-12.
patchtst_pred = patchtst_prediction
transformer_pred = transformer_prediction
#this combines the two step prediction into one 12 step hybrui prediction
hybrid_pred = torch.cat([patchtst_pred, transformer_pred], dim=1)
#combines the targests together
hybrid_targets = torch.cat([patchtst_batch["targets"], transformer_batch["targets"]], dim=1)
# checks 
print("PatchTST prediction shape:", tuple(patchtst_pred.shape))
print("iTransformer prediction shape:", tuple(transformer_pred.shape))
print("Hybrid prediction shape:", tuple(hybrid_pred.shape))
print("Target shape:", tuple(hybrid_targets.shape))

assert tuple(patchtst_pred.shape) == (BATCH_SIZE, SPLIT_OUTPUT_LEN, NUM_FEATURES)
assert tuple(transformer_pred.shape) == (BATCH_SIZE, SPLIT_OUTPUT_LEN, NUM_FEATURES)
assert tuple(hybrid_pred.shape) == (BATCH_SIZE, FULL_OUTPUT_LEN, NUM_FEATURES)
assert tuple(hybrid_targets.shape) == (BATCH_SIZE, FULL_OUTPUT_LEN, NUM_FEATURES)

#find the last saved prediction
def latest_prediction_file(cfg):
    prediction_files = sorted(
        Path(cfg.ckpt_save_dir).rglob("test_results/prediction.npy"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not prediction_files:
        raise FileNotFoundError(
            f"No prediction.npy found under {cfg.ckpt_save_dir}. Train/evaluate this model with save_results=True first."
        )
    return prediction_files[0]

#load the prediction arrays
def load_basicts_array(path, shape):
    # BasicTS writes these files as raw memmaps, even though the file names end in .npy.
    array = np.memmap(path, dtype=np.float32, mode="r", shape=shape)
    return np.asarray(array)

#loads the largets
def load_basicts_prediction_and_targets(cfg, output_len):
    prediction_path = latest_prediction_file(cfg)
    targets_path = prediction_path.parent / "targets.npy"
    if not targets_path.exists():
        raise FileNotFoundError(f"No targets.npy found next to {prediction_path}")

    test_dataset = Builder._build_dataset(cfg, BasicTSMode.TEST)
    shape = (len(test_dataset), output_len, NUM_FEATURES)
    prediction = load_basicts_array(prediction_path, shape)
    targets = load_basicts_array(targets_path, shape)
    return prediction, targets


patchtst_test_pred, patchtst_test_targets = load_basicts_prediction_and_targets(patchtst_cfg, SPLIT_OUTPUT_LEN)
transformer_test_pred, transformer_test_targets = load_basicts_prediction_and_targets(transformer_cfg, SPLIT_OUTPUT_LEN)
#bombines the test predictions
hybrid_test_pred = np.concatenate([patchtst_test_pred, transformer_test_pred], axis=1)
hybrid_test_targets = np.concatenate([patchtst_test_targets, transformer_test_targets], axis=1)

assert patchtst_test_pred.shape == patchtst_test_targets.shape
assert transformer_test_pred.shape == transformer_test_targets.shape
assert patchtst_test_pred.shape == transformer_test_pred.shape
assert hybrid_test_pred.shape == hybrid_test_targets.shape
assert hybrid_test_pred.shape[1] == FULL_OUTPUT_LEN

mixed_output_dir = Path("checkpoints") / RUN_TAG
mixed_output_dir.mkdir(parents=True, exist_ok=True)
hybrid_save_path = mixed_output_dir / "hybrid_split_horizon_patchtst_steps_1_6_itransformer_steps_7_12_ETTh1_96_12_prediction.npy"
np.save(hybrid_save_path, hybrid_test_pred)
print(f"Saved fixed split hybrid prediction: {hybrid_save_path}")

## 12. MAE/MSE Comparison

This cell computes MAE/MSE directly from the saved predictions and targets. The PatchTST is compared only against target steps 1-6, the iTransformer only against target steps 7-12, and the hybrid against the full 12-step target.


In [ ]:
#use math to compute metrics
def compute_metrics(prediction, targets):
    return {
        "MAE": float(np.mean(np.abs(prediction - targets))),
        "MSE": float(np.mean((prediction - targets) ** 2)),
    }

def latest_metrics_file(cfg):
    metrics_files = sorted(
        Path(cfg.ckpt_save_dir).rglob("test_metrics.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not metrics_files:
        raise FileNotFoundError(f"No test_metrics.json found under {cfg.ckpt_save_dir}. Train this model first.")
    return metrics_files[0]


def load_test_metrics(cfg):
    metrics_path = latest_metrics_file(cfg)
    with metrics_path.open("r", encoding="utf-8") as f:
        metrics = json.load(f)
    return metrics.get("overall", metrics)


# Compare the split models, hybrid, and standalone 12-step models.
comparison = {
    "PatchTST split steps 1-6": compute_metrics(patchtst_test_pred, patchtst_test_targets),
    "iTransformer split steps 7-12": compute_metrics(transformer_test_pred, transformer_test_targets),
    "Hybrid split steps 1-12": compute_metrics(hybrid_test_pred, hybrid_test_targets),
    "PatchTST full steps 1-12": load_test_metrics(patchtst_full_cfg),
    "iTransformer full steps 1-12": load_test_metrics(transformer_full_cfg),
}

print("PatchTST target shape:", patchtst_test_targets.shape)
print("iTransformer target shape:", transformer_test_targets.shape)
print("Hybrid target shape:", hybrid_test_targets.shape)
print("PatchTST full 12-step metrics file:", latest_metrics_file(patchtst_full_cfg))
print("iTransformer full 12-step metrics file:", latest_metrics_file(transformer_full_cfg))

for model_name, metrics in comparison.items():
    print(model_name)
    for metric_name in ["MAE", "MSE"]:
        print(f"  {metric_name}: {metrics[metric_name]:.6f}")

## 13. Efficiency Comparison

This cell compares model size and average batch prediction time. The PatchTST timing is for its 6-step prediction, the iTransformer timing is for its 6-step prediction, and the hybrid timing is the sum of running both 6-step models once.


In [ ]:

import time

#counts model size
def count_trainable_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)

#takes one batch and runs the model many times
def time_model_prediction(model, batch, expected_output_len, repeats=50, warmup=5):
    #measure how fast a model makes predictions in one batch
    # check if its on cpu or gpu
    device = next(model.parameters()).device
    # moves input to the same device
    inputs = batch["inputs"].to(device)
    model.eval()

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(inputs)
    #predicts the batch 50 times
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(repeats):
            prediction = model(inputs)
    end = time.perf_counter()

    if isinstance(prediction, dict):
        prediction = prediction["prediction"]
    assert tuple(prediction.shape) == (inputs.size(0), expected_output_len, NUM_FEATURES)
    return (end - start) / repeats


patchtst_efficiency_model = patchtst_cfg.model(patchtst_cfg.model_config)
transformer_efficiency_model = transformer_cfg.model(transformer_cfg.model_config)
patchtst_full_efficiency_model = patchtst_full_cfg.model(patchtst_full_cfg.model_config)
transformer_full_efficiency_model = transformer_full_cfg.model(transformer_full_cfg.model_config)

patchtst_params = count_trainable_parameters(patchtst_efficiency_model)
transformer_params = count_trainable_parameters(transformer_efficiency_model)
patchtst_full_params = count_trainable_parameters(patchtst_full_efficiency_model)
transformer_full_params = count_trainable_parameters(transformer_full_efficiency_model)
hybrid_params = patchtst_params + transformer_params

patchtst_time = time_model_prediction(patchtst_efficiency_model, patchtst_batch, SPLIT_OUTPUT_LEN)
transformer_time = time_model_prediction(transformer_efficiency_model, transformer_batch, SPLIT_OUTPUT_LEN)
patchtst_full_time = time_model_prediction(patchtst_full_efficiency_model, patchtst_batch, FULL_OUTPUT_LEN)
transformer_full_time = time_model_prediction(transformer_full_efficiency_model, transformer_batch, FULL_OUTPUT_LEN)
hybrid_time = patchtst_time + transformer_time

print("PatchTST parameters:", patchtst_params)
print("iTransformer parameters:", transformer_params)
print("Hybrid parameters:", hybrid_params)
print("PatchTST full 12-step parameters:", patchtst_full_params)
print("iTransformer full 12-step parameters:", transformer_full_params)

print(f"PatchTST 6-step avg prediction time: {patchtst_time:.6f} seconds")
print(f"iTransformer 6-step avg prediction time: {transformer_time:.6f} seconds")
print(f"Hybrid avg prediction time: {hybrid_time:.6f} seconds")
print(f"PatchTST full 12-step avg prediction time: {patchtst_full_time:.6f} seconds")
print(f"iTransformer full 12-step avg prediction time: {transformer_full_time:.6f} seconds")

rows = [
    ("PatchTST split 1-6", comparison["PatchTST split steps 1-6"], patchtst_params, patchtst_time),
    ("iTransformer split 7-12", comparison["iTransformer split steps 7-12"], transformer_params, transformer_time),
    ("Hybrid split 1-12", comparison["Hybrid split steps 1-12"], hybrid_params, hybrid_time),
    ("PatchTST full 1-12", comparison["PatchTST full steps 1-12"], patchtst_full_params, patchtst_full_time),
    ("iTransformer full 1-12", comparison["iTransformer full steps 1-12"], transformer_full_params, transformer_full_time),
]

print(f"{'Model':<14} {'MAE':>10} {'MSE':>10} {'Params':>12} {'Avg batch sec':>15}")
print("-" * 65)
for model_name, metrics, params, avg_time in rows:
    print(f"{model_name:<14} {metrics['MAE']:>10.6f} {metrics['MSE']:>10.6f} {params:>12,} {avg_time:>15.6f}")


## 14. Selector-Based Hybrid: Hard and Soft Weighted Selection Without Test Leakage

In this section, we move beyond the fixed split-horizon hybrid (PatchTST steps 1-6, iTransformer steps 7-12) to intelligent selection mechanisms:

1. **Hard Selector**: For each forecast step and feature, choose the model with the lowest validation MAE.
2. **Soft Weighted Selector**: For each forecast step and feature, blend predictions using weights derived from inverse validation MAE.
3. **Temperature Search Softmax Selector**: Test several softmax temperatures on validation MAE, freeze the best one, then evaluate test once.

The important rule: validation data is used to learn the selector, and test data is used only for final evaluation.

In [ ]:
# Generate full 12-step validation and test predictions for selector comparison.
# The selector is learned on validation predictions only, then applied to test predictions.

from types import SimpleNamespace

print("Loading full 12-step model predictions for VAL and TEST...")


def latest_best_checkpoint(cfg):
    metric_name = cfg.target_metric.replace("/", "_")
    checkpoint_name = f"{cfg.model.__name__}_best_val_{metric_name}.pt"
    checkpoint_files = sorted(
        Path(cfg.ckpt_save_dir).rglob(checkpoint_name),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not checkpoint_files:
        raise FileNotFoundError(
            f"No {checkpoint_name} found under {cfg.ckpt_save_dir}. Train the model first."
        )
    return checkpoint_files[0]


def predict_full_model_on_mode(cfg, mode):
    train_dataset = Builder._build_dataset(cfg, BasicTSMode.TRAIN)
    eval_dataset = Builder._build_dataset(cfg, mode)
    eval_loader = DataLoader(eval_dataset, batch_size=cfg.batch_size, shuffle=False)

    scaler = Builder._build_scaler(cfg) if cfg.scaler is not None else None
    if scaler is not None:
        scaler.fit(train_dataset.data)

    model = cfg.model(cfg.model_config)
    checkpoint_path = latest_best_checkpoint(cfg)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    runner = SimpleNamespace(cfg=cfg, scaler=scaler)
    predictions = []
    targets = []

    with torch.no_grad():
        for raw_batch in eval_loader:
            batch = _float_batch(raw_batch)
            batch = cfg.taskflow.preprocess(runner, batch)
            prediction = model(batch["inputs"])
            if isinstance(prediction, dict):
                prediction = prediction["prediction"]
            predictions.append(prediction.cpu().numpy())
            targets.append(batch["targets"].cpu().numpy())

    return np.concatenate(predictions, axis=0), np.concatenate(targets, axis=0), checkpoint_path


patchtst_full_val_pred, patchtst_full_val_targets, patchtst_full_ckpt = predict_full_model_on_mode(
    patchtst_full_cfg,
    BasicTSMode.VAL,
)
transformer_full_val_pred, transformer_full_val_targets, transformer_full_ckpt = predict_full_model_on_mode(
    transformer_full_cfg,
    BasicTSMode.VAL,
)

patchtst_full_test_pred, patchtst_full_test_targets, _ = predict_full_model_on_mode(
    patchtst_full_cfg,
    BasicTSMode.TEST,
)
transformer_full_test_pred, transformer_full_test_targets, _ = predict_full_model_on_mode(
    transformer_full_cfg,
    BasicTSMode.TEST,
)

print(f"PatchTST best checkpoint: {patchtst_full_ckpt}")
print(f"iTransformer best checkpoint: {transformer_full_ckpt}")
print(f"PatchTST VAL pred shape: {patchtst_full_val_pred.shape}")
print(f"iTransformer VAL pred shape: {transformer_full_val_pred.shape}")
print(f"PatchTST TEST pred shape: {patchtst_full_test_pred.shape}")
print(f"iTransformer TEST pred shape: {transformer_full_test_pred.shape}")

assert patchtst_full_val_pred.shape == patchtst_full_val_targets.shape
assert transformer_full_val_pred.shape == transformer_full_val_targets.shape
assert patchtst_full_test_pred.shape == patchtst_full_test_targets.shape
assert transformer_full_test_pred.shape == transformer_full_test_targets.shape
assert patchtst_full_val_pred.shape[1:] == (FULL_OUTPUT_LEN, NUM_FEATURES)
assert patchtst_full_test_pred.shape[1:] == (FULL_OUTPUT_LEN, NUM_FEATURES)

In [ ]:
# Hard Selector: Learn per-step/per-feature choices on VAL, then apply to TEST.

print("\n=== HARD SELECTOR (Validation-Tuned Step-wise Best Model) ===")

patchtst_val_per_step_mae = np.mean(
    np.abs(patchtst_full_val_pred - patchtst_full_val_targets),
    axis=0,
)

transformer_val_per_step_mae = np.mean(
    np.abs(transformer_full_val_pred - transformer_full_val_targets),
    axis=0,
)

print(f"PatchTST validation per-step MAE shape: {patchtst_val_per_step_mae.shape}")
print(f"iTransformer validation per-step MAE shape: {transformer_val_per_step_mae.shape}")

# hard_selector_mask[step, feature] = 0 for PatchTST, 1 for iTransformer.
# This is learned only from validation errors.
hard_selector_mask = (
    transformer_val_per_step_mae < patchtst_val_per_step_mae
).astype(int)

hard_selected_pred = np.where(
    hard_selector_mask[np.newaxis, :, :],
    transformer_full_test_pred,
    patchtst_full_test_pred,
)

hard_selected_targets = patchtst_full_test_targets

print(f"Hard selector mask shape: {hard_selector_mask.shape}")
print(f"Hard-selected TEST prediction shape: {hard_selected_pred.shape}")
print(f"Hard-selected TEST targets shape: {hard_selected_targets.shape}")

assert np.allclose(patchtst_full_test_targets, transformer_full_test_targets)
assert hard_selected_pred.shape == hard_selected_targets.shape

print("\nHard selector learned from VAL: Model choices per step (% iTransformer):")
for step in range(FULL_OUTPUT_LEN):
    pct_transformer = np.mean(hard_selector_mask[step, :]) * 100
    print(f"  Step {step+1}: {pct_transformer:.1f}% iTransformer, {100-pct_transformer:.1f}% PatchTST")

In [ ]:
# Soft Weighted Selector: Learn inverse-MAE weights on VAL, then apply to TEST.

print("\n=== SOFT WEIGHTED SELECTOR (Validation-Tuned Weighted Average) ===")

epsilon = 1e-6

patchtst_weights = 1.0 / (patchtst_val_per_step_mae + epsilon)
transformer_weights = 1.0 / (transformer_val_per_step_mae + epsilon)

total_weights = patchtst_weights + transformer_weights
patchtst_weights_normalized = patchtst_weights / total_weights
transformer_weights_normalized = transformer_weights / total_weights

weighted_pred = (
    patchtst_weights_normalized[np.newaxis, :, :] * patchtst_full_test_pred
    + transformer_weights_normalized[np.newaxis, :, :] * transformer_full_test_pred
)

weighted_targets = patchtst_full_test_targets

print(f"PatchTST validation-derived weights shape: {patchtst_weights_normalized.shape}")
print(f"iTransformer validation-derived weights shape: {transformer_weights_normalized.shape}")
print(f"Weighted TEST prediction shape: {weighted_pred.shape}")
print(f"Weighted TEST targets shape: {weighted_targets.shape}")

assert weighted_pred.shape == weighted_targets.shape

print("\nSoft weights learned from VAL, averaged across features:")
print(f"{'Step':<6} {'PatchTST Weight':<12} {'iTransformer Weight':<18}")
print("-" * 36)

for step in range(FULL_OUTPUT_LEN):
    patchtst_step_weight = np.mean(patchtst_weights_normalized[step, :])
    transformer_step_weight = np.mean(transformer_weights_normalized[step, :])
    print(f"{step+1:<6} {patchtst_step_weight:<12.4f} {transformer_step_weight:<18.4f}")

# Temperature Search Softmax Selector: choose temperature on VAL, then report TEST once.

print("\n=== TEMPERATURE SEARCH SOFTMAX SELECTOR (Validation-Tuned Temperature) ===")

selector_temperatures = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
val_mae_stack = np.stack([patchtst_val_per_step_mae, transformer_val_per_step_mae], axis=0)


def softmax_model_weights_from_mae(mae_stack, temperature):
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    logits = -mae_stack / temperature
    logits = logits - np.max(logits, axis=0, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=0, keepdims=True)


temperature_search_results = []
best_temperature = None
best_temperature_val_mae = float("inf")
best_temperature_weights = None

for temperature in selector_temperatures:
    candidate_weights = softmax_model_weights_from_mae(val_mae_stack, temperature)
    candidate_val_pred = (
        candidate_weights[0][np.newaxis, :, :] * patchtst_full_val_pred
        + candidate_weights[1][np.newaxis, :, :] * transformer_full_val_pred
    )
    candidate_val_mae = float(np.mean(np.abs(candidate_val_pred - patchtst_full_val_targets)))
    temperature_search_results.append({
        "temperature": float(temperature),
        "validation_mae": candidate_val_mae,
    })

    if candidate_val_mae < best_temperature_val_mae:
        best_temperature = float(temperature)
        best_temperature_val_mae = candidate_val_mae
        best_temperature_weights = candidate_weights

temperature_softmax_pred = (
    best_temperature_weights[0][np.newaxis, :, :] * patchtst_full_test_pred
    + best_temperature_weights[1][np.newaxis, :, :] * transformer_full_test_pred
)
temperature_softmax_targets = patchtst_full_test_targets
temperature_patchtst_weights = best_temperature_weights[0]
temperature_transformer_weights = best_temperature_weights[1]

print(f"Candidate temperatures: {selector_temperatures}")
print(f"Selected temperature by VAL MAE: {best_temperature:.2f}")
print(f"Best validation MAE: {best_temperature_val_mae:.6f}")
print("\nTemperature search validation MAE:")
print(f"{'Temperature':<14} {'VAL MAE':>12}")
print("-" * 28)
for result in temperature_search_results:
    print(f"{result['temperature']:<14.2f} {result['validation_mae']:>12.6f}")

print("\nSelected softmax weights learned from VAL, averaged across features:")
print(f"{'Step':<6} {'PatchTST Weight':<12} {'iTransformer Weight':<18}")
print("-" * 36)
for step in range(FULL_OUTPUT_LEN):
    patchtst_step_weight = np.mean(temperature_patchtst_weights[step, :])
    transformer_step_weight = np.mean(temperature_transformer_weights[step, :])
    print(f"{step+1:<6} {patchtst_step_weight:<12.4f} {transformer_step_weight:<18.4f}")

assert temperature_softmax_pred.shape == temperature_softmax_targets.shape

In [ ]:
# Comprehensive metrics table.
# The selector hybrids were tuned on VAL and are evaluated here on TEST only.

print("\n=== COMPREHENSIVE TEST METRICS COMPARISON ===\n")

all_comparison = {
    "PatchTST split steps 1-6": compute_metrics(patchtst_test_pred, patchtst_test_targets),
    "iTransformer split steps 7-12": compute_metrics(transformer_test_pred, transformer_test_targets),
    "Fixed split hybrid 1-12": compute_metrics(hybrid_test_pred, hybrid_test_targets),
    "PatchTST full steps 1-12": compute_metrics(patchtst_full_test_pred, patchtst_full_test_targets),
    "iTransformer full steps 1-12": compute_metrics(transformer_full_test_pred, transformer_full_test_targets),
    "Hard selector hybrid 1-12": compute_metrics(hard_selected_pred, hard_selected_targets),
    "Soft weighted hybrid 1-12": compute_metrics(weighted_pred, weighted_targets),
    "Temperature search softmax hybrid 1-12": compute_metrics(temperature_softmax_pred, temperature_softmax_targets),
}

print(f"{'Model':<35} {'MAE':>12} {'MSE':>12}")
print("-" * 60)
for model_name in [
    "PatchTST split steps 1-6",
    "iTransformer split steps 7-12",
    "Fixed split hybrid 1-12",
    "PatchTST full steps 1-12",
    "iTransformer full steps 1-12",
    "Hard selector hybrid 1-12",
    "Soft weighted hybrid 1-12",
    "Temperature search softmax hybrid 1-12",
]:
    metrics = all_comparison[model_name]
    mae = metrics["MAE"]
    mse = metrics["MSE"]
    print(f"{model_name:<35} {mae:>12.6f} {mse:>12.6f}")

best_mae_model = min(all_comparison.items(), key=lambda x: x[1]["MAE"])
best_mse_model = min(all_comparison.items(), key=lambda x: x[1]["MSE"])

print("\n" + "=" * 60)
print(f"Best TEST MAE: {best_mae_model[0]:<30} {best_mae_model[1]['MAE']:.6f}")
print(f"Best TEST MSE: {best_mse_model[0]:<30} {best_mse_model[1]['MSE']:.6f}")

In [ ]:
# Save selector hybrid predictions to the mixed run folder in checkpoints/.

print("\n=== SAVING LEAKAGE-SAFE SELECTOR PREDICTIONS ===\n")

mixed_output_dir = Path("checkpoints") / RUN_TAG
mixed_output_dir.mkdir(parents=True, exist_ok=True)

hard_selector_save_path = mixed_output_dir / "hard_selector_val_tuned_hybrid_patchtst_itransformer_ETTh1_96_12_prediction.npy"
np.save(hard_selector_save_path, hard_selected_pred)
print(f"Saved hard selector predictions: {hard_selector_save_path}")
print(f"  Shape: {hard_selected_pred.shape}")

weighted_selector_save_path = mixed_output_dir / "soft_weighted_val_tuned_hybrid_patchtst_itransformer_ETTh1_96_12_prediction.npy"
np.save(weighted_selector_save_path, weighted_pred)
print(f"Saved soft weighted predictions: {weighted_selector_save_path}")
print(f"  Shape: {weighted_pred.shape}")

temperature_selector_save_path = mixed_output_dir / "temperature_search_softmax_val_tuned_hybrid_patchtst_itransformer_ETTh1_96_12_prediction.npy"
np.save(temperature_selector_save_path, temperature_softmax_pred)
print(f"Saved temperature-search softmax predictions: {temperature_selector_save_path}")
print(f"  Shape: {temperature_softmax_pred.shape}")

weights_info = {
    "selector_fit_split": "validation",
    "final_evaluation_split": "test",
    "patchtst_weights": patchtst_weights_normalized.tolist(),
    "itransformer_weights": transformer_weights_normalized.tolist(),
    "temperature_search": {
        "candidate_temperatures": selector_temperatures,
        "validation_mae_by_temperature": temperature_search_results,
        "best_temperature": best_temperature,
        "best_validation_mae": best_temperature_val_mae,
        "patchtst_weights": temperature_patchtst_weights.tolist(),
        "itransformer_weights": temperature_transformer_weights.tolist(),
    },
    "hard_selector_mask": hard_selector_mask.tolist(),
    "description": "Hard selector mask: 0=PatchTST, 1=iTransformer. Soft weights are normalized inverse validation MAE per step and feature. Temperature search uses validation MAE to choose one softmax temperature, then evaluates that frozen choice on test once.",
}
weights_save_path = mixed_output_dir / "selector_metadata_val_tuned_ETTh1_96_12.json"
with weights_save_path.open("w", encoding="utf-8") as f:
    json.dump(weights_info, f, indent=2)
print(f"Saved selector metadata: {weights_save_path}")

hybrid_metrics_save_path = mixed_output_dir / "hybrid_metrics_ETTh1_96_12.json"
with hybrid_metrics_save_path.open("w", encoding="utf-8") as f:
    json.dump(all_comparison, f, indent=2)
print(f"Saved hybrid metrics: {hybrid_metrics_save_path}")

print("All leakage-safe selector predictions and metadata saved successfully.")

In [ ]:
# Summary and interpretation of selector approaches

print("\n" + "="*70)
print("LEAKAGE-SAFE SELECTOR-BASED HYBRID SUMMARY")
print("="*70)

print("""
Four hybrid approaches have been compared:

1. FIXED SPLIT HYBRID
   - PatchTST forecasts steps 1-6
   - iTransformer forecasts steps 7-12
   - Simple concatenation; uses pre-trained split-horizon models

2. HARD SELECTOR HYBRID
   - For each forecast step and feature, choose the model with the lowest validation MAE
   - The selector mask is frozen before test evaluation
   - File: hard_selector_val_tuned_hybrid_patchtst_itransformer_ETTh1_96_12_prediction.npy

3. SOFT WEIGHTED HYBRID
   - For each forecast step and feature, weight-average both model predictions
   - Weights are based on inverse validation MAE, not test MAE
   - File: soft_weighted_val_tuned_hybrid_patchtst_itransformer_ETTh1_96_12_prediction.npy

4. TEMPERATURE SEARCH SOFTMAX HYBRID
   - Tests temperatures [0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0] on validation MAE
   - Freezes the best validation temperature before test evaluation
   - File: temperature_search_softmax_val_tuned_hybrid_patchtst_itransformer_ETTh1_96_12_prediction.npy

Leakage control:
- Validation data chooses the selector mask, weights, and temperature
- Test data is used only once for final metrics
- Do not recompute selector choices from test targets
""")

print("="*70)
print("Selector-based hybrid implementation complete.")
print("="*70)